# Supply Chain Risk Modelling — Phase 2: Feature Engineering & Modelling

The WRI gives raw material — 248 columns describing how exposed and vulnerable each country is. But none of those columns directly answer what a logistics analyst would ask: *how likely is a disruption here, and how bad would it be for the network?*

This notebook builds the answer in layers. Five engineered features compress the WRI's sub-indicators into interpretable risk dimensions, each one a deliberate choice about what actually drives supply chain failure. A sixth feature captures something the WRI doesn't measure at all: how irreplaceable a country is as a routing node. Those features then feed into a composite score, an ML classification layer, and an LSTM forecasting model that adds a sequence-aware prediction on top.

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load the Phase 1 outputs
DATA_PATH = "./"
wri = pd.read_csv(f"{DATA_PATH}wri_full_clean.csv")

print(f"Loaded: {wri.shape[0]} rows × {wri.shape[1]} cols")
print(f"Scope: {wri['country'].nunique()} countries, {wri['year'].min()}-{wri['year'].max()}")

Loaded: 5018 rows × 248 cols
Scope: 193 countries, 2000-2025


## Feature 1 — Natural Hazard Exposure Score (NHES)

The WRI's Exposure dimension aggregates seven hazard types, but for logistics purposes they carry very different operational weight. Earthquakes and cyclones close ports overnight; sea level rise is a decade-scale concern. The NHES reweighs those seven domains by disruption severity rather than treating them equally — the explicit assumption being that sudden, total infrastructure failures matter more to a routing model than slow-onset threats.

In [5]:
# WRI domain inputs — already on 0-100 scale, higher = more exposed
#   ei_01 Earthquakes:       sudden total failure, hardest to route around
#   ei_02 Tsunamis:          destructive but geographically narrow
#   ei_03 Coastal flooding:  hits port infrastructure directly
#   ei_04 Riverine flooding: cuts inland road and rail networks
#   ei_05 Cyclones:          widespread multi-day shutdown
#   ei_06 Droughts:          slow-onset, logistics can partially absorb
#   ei_07 Sea level rise:    decade-scale, outside near-term routing horizon
#
# Weights reflect logistics disruption severity — sudden, total failures
# score higher than slow-onset threats that planning can partially anticipate.
# Weights reflect OPERATIONAL DISRUPTION to logistics, not human welfare.

NHES_WEIGHTS = {
    'ei_01': 0.20,  # Earthquakes: sudden total failure, highest weight
    'ei_02': 0.10,  # Tsunamis: devastating but geographically narrow
    'ei_03': 0.15,  # Coastal flooding: directly hits port infrastructure
    'ei_04': 0.20,  # Riverine flooding: cuts inland networks for weeks
    'ei_05': 0.20,  # Cyclones: multi-day widespread shutdown
    'ei_06': 0.10,  # Droughts: slow-onset, lower logistics urgency
    'ei_07': 0.05,  # Sea level rise: long-term, outside 3-5yr horizon
}

# Verify weights sum to 1.0 (a sanity check you should always include)
assert abs(sum(NHES_WEIGHTS.values()) - 1.0) < 1e-9, "NHES weights must sum to 1.0"

# Compute weighted sum
wri['nhes'] = sum(wri[col] * weight for col, weight in NHES_WEIGHTS.items())

# Quick sanity check: high-exposure vs low-exposure countries
print("=" * 70)
print("FEATURE 1: NHES — Natural Hazard Exposure Score")
print("=" * 70)
latest = wri[wri['year'] == wri['year'].max()]
print("\nHighest NHES (most exposed to logistics-disrupting hazards):")
print(latest.nlargest(10, 'nhes')[['country', 'nhes', 'ei_01', 'ei_04', 'ei_05']].to_string(index=False))
print("\nLowest NHES:")
print(latest.nsmallest(5, 'nhes')[['country', 'nhes']].to_string(index=False))

FEATURE 1: NHES — Natural Hazard Exposure Score

Highest NHES (most exposed to logistics-disrupting hazards):
                 country  nhes  ei_01  ei_04  ei_05
                   China 69.36  61.24  78.57  74.98
                   Japan 65.05  81.67  51.10  85.03
             Philippines 59.42  77.65  41.48  89.72
                  Mexico 53.13  50.99  52.30  74.21
               Indonesia 50.11  71.01  53.27   2.74
United States of America 47.36  54.87  48.10  58.28
                    Peru 46.16  66.03  44.03   0.01
                   India 45.96  45.13  72.66   2.90
                Viet Nam 44.42   2.21  82.24   3.28
                 Myanmar 42.41  57.08  71.05   2.82

Lowest NHES:
              country  nhes
Sao Tome and Principe  0.04
              Andorra  0.05
               Monaco  0.38
           San Marino  0.43
           Cape Verde  2.08


## Feature 2 — Governance and Stability Score (GSS)

In practice, governance quality is the strongest single predictor of logistics reliability — more so than physical hazards alone. Weak institutions compound every other risk: a flood that disrupts Singapore for two weeks can shut down a poorly-governed corridor for two months, because the government can't coordinate emergency response, customs continues to back up, and infrastructure repairs stall. The GSS draws on WRI's state and government sub-indicators, normalized and oriented so that higher always means more governance-related risk.

In [7]:
gov_base_cols = ['ci_03a_base', 'ci_03b_base', 'ci_04a_base', 'ci_04b_base']

# Normalise each governance column to 0-100 using global min/max across all country-years.
# Global rather than per-year normalisation keeps scores comparable over time.

for col in gov_base_cols:
    col_min = wri[col].min()
    col_max = wri[col].max()
    wri[f'{col}_scaled'] = (wri[col] - col_min) / (col_max - col_min) * 100

# Two sub-composites reduce multicollinearity — the four raw indicators
# are highly correlated, so averaging within groups before combining.
# Democratic Quality:     corruption control + rule of law
# Operational Governance: government effectiveness + political stability

wri['gss_democratic'] = (wri['ci_03a_base_scaled'] + wri['ci_03b_base_scaled']) / 2

wri['gss_operational'] = (wri['ci_04a_base_scaled'] + wri['ci_04b_base_scaled']) / 2

# Step 3: Blend the two sub-composites
# Operational governance gets slightly more weight because it directly affects day-to-day logistics (customs, port admin, crisis response)

wri['gss'] = 0.45 * wri['gss_democratic'] + 0.55 * wri['gss_operational']

# IMPORTANT: At this point, GSS is "higher = BETTER governance"(because the raw WGI scale goes from bad→good and we preserved direction)
# We'll invert when building the composite route risk score later. For now, we KEEP this direction so the feature is interpretable on its own.

print("=" * 70)
print("FEATURE 2: GSS — Governance and Stability Score")
print("(Higher = BETTER governance = LOWER risk)")
print("=" * 70)
latest = wri[wri['year'] == wri['year'].max()]
print("\nBest governance (highest GSS):")
print(latest.nlargest(10, 'gss')[['country', 'gss', 'gss_democratic', 'gss_operational']].to_string(index=False))
print("\nWorst governance (lowest GSS):")
print(latest.nsmallest(10, 'gss')[['country', 'gss', 'gss_democratic', 'gss_operational']].to_string(index=False))

FEATURE 2: GSS — Governance and Stability Score
(Higher = BETTER governance = LOWER risk)

Best governance (highest GSS):
      country   gss  gss_democratic  gss_operational
    Singapore 93.07           93.59            92.65
      Denmark 91.25           94.41            88.66
  Switzerland 90.28           91.52            89.27
  New Zealand 89.06           86.08            91.50
Liechtenstein 88.97           82.73            94.07
      Finland 88.77           89.80            87.93
       Norway 88.68           89.17            88.28
   Luxembourg 88.62           88.16            89.00
       Monaco 88.49           86.70            89.96
      Iceland 85.70           80.15            90.24

Worst governance (lowest GSS):
                     country   gss  gss_democratic  gss_operational
        Syrian Arab Republic  9.79            7.78            11.42
                     Somalia  9.98            6.03            13.22
                       Yemen 10.24            4.00         

## Feature 3 — Infrastructure Resilience Score (IRS)

Physical infrastructure determines how quickly a corridor recovers after a shock, independent of governance quality. A country can have functional institutions but crumbling roads and intermittent power — or vice versa. The IRS draws on WRI's services and connectivity indicators, weighted towards electricity and communications capacity since modern port operations depend on both far more than on water infrastructure.

In [9]:
# _Norm columns are already 0-100, with higher = worse (more susceptible / less access).
# That direction is exactly what we want — HIGH IRS = POOR infrastructure = HIGH risk.

# Weights by logistics relevance:
#   Electricity and comms dominate — essential for modern port operations
#   Water/sanitation serve as general infrastructure quality proxies
#   Mobile weighted above broadband for developing-country applicability

IRS_WEIGHTS = {
    'si_06a_norm': 0.25,  # Electricity access — critical for port ops
    'si_06b_norm': 0.10,  # Clean cooking fuels — energy modernization proxy
    'si_07a_norm': 0.15,  # Fixed broadband — logistics coordination
    'si_07b_norm': 0.20,  # Mobile cellular — tracking, last-mile comms
    'si_05a_norm': 0.15,  # Water access — general infra quality proxy
    'si_05b_norm': 0.15,  # Sanitation — general infra quality proxy
}

assert abs(sum(IRS_WEIGHTS.values()) - 1.0) < 1e-9, "IRS weights must sum to 1.0"

wri['irs'] = sum(wri[col] * weight for col, weight in IRS_WEIGHTS.items())

# Direction: Higher IRS = worse infrastructure = higher risk

print("=" * 70)
print("FEATURE 3: IRS — Infrastructure Resilience Score")
print("(Higher = WORSE infrastructure = HIGHER risk)")
print("=" * 70)
latest = wri[wri['year'] == wri['year'].max()]
print("\nWorst infrastructure (highest IRS = highest risk):")
print(latest.nlargest(10, 'irs')[['country', 'irs', 'si_06a_norm', 'si_07b_norm']].to_string(index=False))
print("\nBest infrastructure (lowest IRS = lowest risk):")
print(latest.nsmallest(5, 'irs')[['country', 'irs']].to_string(index=False))

FEATURE 3: IRS — Infrastructure Resilience Score
(Higher = WORSE infrastructure = HIGHER risk)

Worst infrastructure (highest IRS = highest risk):
                     country   irs  si_06a_norm  si_07b_norm
                 South Sudan 85.48       100.00        75.73
    Central African Republic 84.58        76.26        87.45
                        Chad 81.26        83.27        69.54
Democratic Republic of Congo 78.42        67.89        81.55
                     Burundi 77.91        88.92        71.32
                       Niger 76.02        73.72        73.91
            Papua New Guinea 75.05        71.53        82.72
                     Liberia 72.83        66.32        89.84
                      Malawi 70.76        79.33        72.82
                  Madagascar 70.59        62.30        64.43

Best infrastructure (lowest IRS = lowest risk):
             country   irs
   Republic of Korea 13.27
United Arab Emirates 14.06
               Japan 14.53
             Andorra 15.0

## Feature 4 — Economic Fragility Score (EFS)

Economic fragility determines how much fiscal buffer a country has to restore logistics capacity when something goes wrong. Low-income, high-aid-dependency countries have less capacity to fund emergency repairs, clear port backlogs, or subsidise carrier rerouting. Inflation volatility gets meaningful weight here because it makes freight contracts unpredictable in ways that directly affect supply chain planning — not just theoretical macro risk.

In [11]:
# All _Norm inputs — higher = more economically fragile.
#   si_03a_norm: GNI per capita  (high norm = low income)
#   si_04a_norm: Aid dependency   (high norm = high reliance on external support)
#   ai_05b_norm: Price instability (high norm = high inflation volatility)

# Weights: GNI gets the most weight as the broadest proxy for fiscal capacity.
# Inflation volatility scores meaningfully because it makes freight contracts
# unpredictable — not just a theoretical macro risk, a practical one.
# Aid dependency is a structural signal but less directly operational

EFS_WEIGHTS = {
    'si_03a_norm': 0.40,  # GNI per capita — overall economic capacity
    'ai_05b_norm': 0.35,  # Inflation instability — freight cost uncertainty
    'si_04a_norm': 0.25,  # Aid dependency — structural fragility signal
}

assert abs(sum(EFS_WEIGHTS.values()) - 1.0) < 1e-9

wri['efs'] = sum(wri[col] * weight for col, weight in EFS_WEIGHTS.items())

print("=" * 70)
print("FEATURE 4: EFS — Economic Fragility Score")
print("(Higher = MORE economically fragile = HIGHER risk)")
print("=" * 70)
latest = wri[wri['year'] == wri['year'].max()]
print("\nMost economically fragile:")
print(latest.nlargest(10, 'efs')[['country', 'efs', 'si_03a_norm', 'ai_05b_norm']].to_string(index=False))
print("\nLeast economically fragile:")
print(latest.nsmallest(5, 'efs')[['country', 'efs']].to_string(index=False))

FEATURE 4: EFS — Economic Fragility Score
(Higher = MORE economically fragile = HIGHER risk)

Most economically fragile:
                 country   efs  si_03a_norm  ai_05b_norm
             South Sudan 82.08        89.84        85.59
                 Burundi 78.92       100.00        71.80
    Syrian Arab Republic 76.11        65.76        84.04
                Zimbabwe 74.97        69.12       100.00
Central African Republic 74.20        93.27        57.16
                  Malawi 73.71        81.55        76.40
                   Sudan 73.32        75.09        87.45
                 Lebanon 72.45        55.18        89.84
             Afghanistan 72.19        77.11        64.92
            Sierra Leone 70.91        69.97        79.55

Least economically fragile:
          country   efs
            Qatar 15.01
Brunei Darussalam 20.13
            China 22.77
          Ireland 23.70
      Switzerland 23.86


## Feature 5 — Recent Shock Burden Score (RSBS)

A country's current resilience is shaped heavily by how many recent crises it has had to absorb. The RSBS captures the five-year rolling burden of disaster deaths and conflict casualties — not as background context, but as a direct proxy for depleted response capacity. Conflict deaths receive higher weight because armed conflict creates total, unpredictable logistics shutdowns with no rerouting alternative, unlike natural disasters where parallel infrastructure often exists.

In [13]:
# Both inputs are _Norm columns — higher = worse.
#   ci_01a_norm: disaster-affected population (5-year rolling average)
#   ci_02a_norm: conflict deaths (5-year rolling average)

# Conflict deaths get higher weight: armed conflict creates total, unpredictable
# logistics shutdowns with no workaround. Floods are devastating but you can
# often reroute; active conflict zones are simply off the table.

RSBS_WEIGHTS = {
    'ci_01a_norm': 0.40,  # Disaster burden — depletes response capacity
    'ci_02a_norm': 0.60,  # Conflict burden — most severe logistics blocker
}

assert abs(sum(RSBS_WEIGHTS.values()) - 1.0) < 1e-9

wri['rsbs'] = sum(wri[col] * weight for col, weight in RSBS_WEIGHTS.items())

print("=" * 70)
print("FEATURE 5: RSBS — Recent Shock Burden Score")
print("(Higher = MORE active crisis = HIGHER risk)")
print("=" * 70)
latest = wri[wri['year'] == wri['year'].max()]
print("\nHighest shock burden:")
print(latest.nlargest(10, 'rsbs')[['country', 'rsbs', 'ci_01a_norm', 'ci_02a_norm']].to_string(index=False))
print("\nLowest shock burden:")
print(latest.nsmallest(5, 'rsbs')[['country', 'rsbs']].to_string(index=False))

FEATURE 5: RSBS — Recent Shock Burden Score
(Higher = MORE active crisis = HIGHER risk)

Highest shock burden:
                     country  rsbs  ci_01a_norm  ci_02a_norm
                    Ethiopia 85.07        62.67       100.00
        Syrian Arab Republic 79.17        75.13        81.87
                 Afghanistan 75.72        57.30        88.00
                 Philippines 73.54        90.93        61.95
                       India 72.63        86.31        63.51
                     Somalia 70.57        69.15        71.52
                       Yemen 69.07        56.27        77.60
                     Nigeria 68.41        67.29        69.16
                    Pakistan 65.29        83.09        53.43
Democratic Republic of Congo 63.17        60.11        65.21

Lowest shock burden:
            country  rsbs
            Andorra  0.01
Antigua and Barbuda  0.01
            Bahrain  0.01
            Bahamas  0.01
  Brunei Darussalam  0.01


## Feature 6 — Supply Chain Concentration Risk (SCCR)

This is the dimension the WRI dataset fundamentally cannot provide. The WRI scores how dangerous a country is to *operate in* — but for supply chain routing, what matters equally is how catastrophic it would be if that country became unavailable. Singapore scores low on every WRI dimension correctly: low hazard exposure, excellent governance, strong infrastructure. But it handles roughly 37 million TEUs annually and controls access to the Strait of Malacca, one of two viable deep-water passages between the Indian and Pacific Oceans. A disruption there doesn't just affect Singapore — it affects a quarter of global trade.

SCCR is built as a manual scoring table weighted by container throughput volume (40%), chokepoint control (40%), and a substitutability penalty (20% — how easily can traffic divert elsewhere?). Crucially, it's applied as a *post-composite multiplier* rather than a weighted input to the CCRS. The reason: SCCR measures the *consequence* of disruption, while the other five features measure *probability*. Averaging an impact metric with probability metrics doesn't produce a meaningful composite — it blurs two analytically distinct questions into one number.

In [15]:
# WRI Exposure measures natural hazard intensity — it says nothing about
# how much global trade flows through a country or how easily that traffic
# could divert elsewhere. Singapore correctly scores low on NHES (it's not
# in a seismic or cyclone belt), but it handles ~37M TEU/year and sits at the
# Strait of Malacca. SCCR fills that gap by capturing strategic irreplaceability.

SCCR_SCORES = {
    # Scores: 0–100, higher = more strategically critical
    # Basis: container throughput volume (40%) + chokepoint control (40%)
    #        + substitutability penalty (20%) — how easily can traffic divert?
    'EGY': 92,  # Suez Canal — ~12% of global trade, no viable bypass
    'SGP': 90,  # Malacca gateway + 37M TEU, second-busiest port globally
    'PAN': 85,  # Panama Canal — transpacific trade backbone
    'DJI': 78,  # Bab-el-Mandeb, sole access to Red Sea from Gulf of Aden
    'YEM': 72,  # Bab-el-Mandeb (opposite shore) — same chokepoint, fragile side
    'MYS': 68,  # Malacca strait + Port Klang, partially overlaps with SGP
    'TUR': 65,  # Bosphorus + Dardanelles — sole Black Sea outlet
    'OMN': 60,  # Strait of Hormuz — ~20% of global petroleum trade
    'IRN': 60,  # Strait of Hormuz (Iran side) — same chokepoint
    'NLD': 58,  # Rotterdam — largest European port, Rhine inland gateway
    'CHN': 55,  # Shanghai/Shenzhen — largest container port by volume
    'USA': 45,  # LA/Long Beach — primary transpacific entry
    'DEU': 40,  # Hamburg — Northern European distribution hub
    'BEL': 38,  # Antwerp — significant but Rotterdam-adjacent
    'HKG': 50,  # Hong Kong — declining but still strategically relevant
    'KOR': 42,  # Busan — major Northeast Asian transhipment hub
    'JPN': 40,  # Tokyo/Yokohama — large but mostly domestic-facing
    'MAR': 35,  # Strait of Gibraltar — important but partial bypass exists
    'ESP': 32,  # Algeciras — Gibraltar corridor, Spain side
    'IDN': 38,  # Lombok/Sunda straits — Malacca alternatives
    'AUS': 25,  # Significant but geographically peripheral
    # All countries not listed receive a default near-zero score
}

# Countries not in the table get 5 rather than 0 — every port has
# some minimum strategic value, and 0 would suggest complete irrelevance.
wri['sccr'] = wri['iso3'].map(SCCR_SCORES).fillna(5)

# Quick sanity check: confirm top nodes look right and Singapore's profile is clear
print("=" * 70)
print("FEATURE 6: SCCR — Supply Chain Concentration Risk")
print("Higher = more strategically critical node in global trade network")
print("=" * 70)
latest = wri[wri['year'] == wri['year'].max()]
print("\nTop 15 most critical supply chain nodes:")
print(
    latest.nlargest(15, 'sccr')[['country', 'iso3', 'sccr', 'nhes', 'w']]
    .to_string(index=False)
)
print("\nSingapore's profile (the key validation case):")
sgp = latest[latest['iso3'] == 'SGP']
print(f"  NHES (natural hazard exposure):  {sgp['nhes'].values[0]:.1f}  ← correctly low")
print(f"  SCCR (supply chain criticality): {sgp['sccr'].values[0]:.1f}  ← now captured")
print(f"  WRI overall score (w):           {sgp['w'].values[0]:.1f}  ← WRI composite (pre-CCRS)")

FEATURE 6: SCCR — Supply Chain Concentration Risk
Higher = more strategically critical node in global trade network

Top 15 most critical supply chain nodes:
                   country iso3  sccr  nhes     w
                     Egypt  EGY 92.00 42.40 18.91
                 Singapore  SGP 90.00  2.48  0.67
                    Panama  PAN 85.00 28.25 18.25
                  Djibouti  DJI 78.00 22.16 10.03
                     Yemen  YEM 72.00 28.63 24.83
                  Malaysia  MYS 68.00 23.48 14.28
                    Turkey  TUR 65.00 31.54 14.62
Iran (Islamic Republic of)  IRN 60.00 33.56 16.30
                      Oman  OMN 60.00 16.21  7.84
               Netherlands  NLD 58.00 31.30  4.33
                     China  CHN 55.00 69.36 30.62
  United States of America  USA 45.00 47.36 23.09
         Republic of Korea  KOR 42.00 40.55 11.01
                   Germany  DEU 40.00 25.86  4.28
                     Japan  JPN 40.00 65.05 24.81

Singapore's profile (the key validation c

## Directionality Check and Composite Scoring

Before combining the five features, they all need to point the same way: higher = riskier. Four of the five already do. The exception is GSS, where raw WRI governance scores run "higher = better governance." Forgetting to invert this produces a composite that rates Somalia as safer than Singapore — the model runs without error and the numbers look plausible until you spot-check a specific country. The inversion (`gss_risk = 100 - gss`) is a single line but it's the most consequential one in the whole feature engineering section.

The resulting CCRS is the probability layer only — it answers "how fragile is this country as an operating environment?" SCCR gets applied afterward as a multiplier, keeping the two analytical questions cleanly separated.

In [17]:
# Direction check before compositing — everything must point the same way (higher = riskier):
#   nhes     → already correct
#   gss      → NEEDS INVERSION: raw WRI governance scores higher = better governance
#   irs      → already correct
#   efs      → already correct
#   rsbs     → already correct

# Flip GSS so that high score = high governance risk.
# Miss this and the composite rates Somalia safer than Singapore.
wri['gss_risk'] = 100 - wri['gss']

# The five fragility/probability features. SCCR is intentionally excluded here —
# it measures disruption consequence, not probability. Combining the two in a
# weighted average conflates analytically distinct questions.
# RISK_FEATURES: the five probability-of-disruption dimensions.
# SCCR is intentionally excluded here — it measures CONSEQUENCE of disruption,
# not probability. Mixing an impact metric into a probability-weighted composite
# produces analytically incorrect results. SCCR is applied separately as a
# post-composite multiplier after ccrs_compound is computed (see Section 3).
RISK_FEATURES = ['nhes', 'gss_risk', 'irs', 'efs', 'rsbs']

# CCRS = weighted composite of the five fragility dimensions.
# Think of it as the probability layer: how likely is disruption here?
CCRS_WEIGHTS = {
    'nhes':     0.20,  # Natural hazard exposure
    'gss_risk': 0.25,  # Governance risk (highest weight — most predictive for logistics)
    'irs':      0.20,  # Infrastructure resilience
    'efs':      0.15,  # Economic fragility
    'rsbs':     0.20,  # Recent shock burden
    # SCCR excluded by design — applied separately as a multiplier to keep
    # probability and impact cleanly separated.
}

assert abs(sum(CCRS_WEIGHTS.values()) - 1.0) < 1e-9

wri['ccrs'] = sum(wri[feat] * weight for feat, weight in CCRS_WEIGHTS.items())

print('=' * 70)
print('COMPOSITE COUNTRY RISK SCORE (CCRS) — Probability Layer')
print('Measures fragility/probability of disruption across 5 dimensions')
print('=' * 70)
latest = wri[wri['year'] == wri['year'].max()]
print('\nTop 15 riskiest countries (by fragility):')
top15_cols = ['country', 'ccrs', 'nhes', 'gss_risk', 'irs', 'efs', 'rsbs']
print(latest.nlargest(15, 'ccrs')[top15_cols].to_string(index=False))

print('\nTop 10 lowest-fragility countries:')
print(latest.nsmallest(10, 'ccrs')[top15_cols].to_string(index=False))

# Singapore validation: WRI Exposure ≠ Supply Chain Exposure
# Singapore's low ccrs is CORRECT — it has low fragility as an operating environment.
# Its strategic supply chain risk is captured separately via SCCR (see Section 3).
sgp = latest[latest['country'] == 'Singapore']
if len(sgp):
    print(f'\nSINGAPORE — Probability Layer Only (ccrs_final adds SCCR impact):')
    print(f'  NHES (hazard exposure):    {sgp["nhes"].values[0]:.1f}  ← correctly low per WRI')
    print(f'  GSS_risk (governance):     {sgp["gss_risk"].values[0]:.1f}  ← excellent governance')
    print(f'  CCRS (fragility score):    {sgp["ccrs"].values[0]:.1f}  ← low fragility, correct')
    print(f'  Note: ccrs_final (after SCCR multiplier) will be higher,')
    print(f'        reflecting Malacca gateway criticality. See Section 3.')


COMPOSITE COUNTRY RISK SCORE (CCRS) — Probability Layer
Measures fragility/probability of disruption across 5 dimensions

Top 15 riskiest countries (by fragility):
                     country  ccrs  nhes  gss_risk   irs   efs  rsbs
                     Somalia 65.98 26.32     90.02 67.29 70.90 70.57
Democratic Republic of Congo 65.02 27.78     82.41 78.42 70.29 63.17
                       Yemen 63.75 28.63     89.76 57.45 68.56 69.07
        Syrian Arab Republic 63.67 22.78     90.21 46.53 76.11 79.17
                 Afghanistan 62.45 20.88     84.33 56.09 72.19 75.72
                    Ethiopia 61.68 20.27     66.65 68.75 68.02 85.07
                 South Sudan 60.73 19.64     65.12 85.48 82.08 55.54
    Central African Republic 60.55 14.40     83.11 84.58 74.20 44.25
                        Chad 59.36 17.75     76.99 81.26 64.56 53.11
                     Myanmar 59.16 42.41     81.44 47.12 63.42 56.91
                       Sudan 58.35 20.16     86.14 58.78 73.32 50.15
        

## Temporal Features

A static risk score describes where a country stands today. Temporal features describe trajectory — and trajectory often matters more than level. A country scoring 50 after climbing from 30 over five years is a fundamentally different routing bet than one that was 65 and has been recovering. These rolling calculations require the data sorted by country and year before groupby operations; getting this order wrong produces silently incorrect rolling statistics without raising any errors.

In [19]:
# Must be sorted by country then year before any rolling operations.
# Getting this wrong produces silently incorrect rolling statistics.
wri = wri.sort_values(['iso3', 'year']).reset_index(drop=True)

# --- Year-over-year change ---
# Simplest temporal feature: how much did risk shift from the previous year?
wri['ccrs_yoy'] = wri.groupby('iso3')['ccrs'].diff()
wri['ccrs_yoy_pct'] = wri.groupby('iso3')['ccrs'].pct_change() * 100

# --- Rolling volatility (3-year and 5-year windows) ---
# Standard deviation of CCRS over a rolling window.
# High volatility = unpredictable risk environment, harder to plan around.
# min_periods prevents features from being built on too few observations.
wri['ccrs_vol_3y'] = (
    wri.groupby('iso3')['ccrs']
    .transform(lambda x: x.rolling(window=3, min_periods=2).std())
)
wri['ccrs_vol_5y'] = (
    wri.groupby('iso3')['ccrs']
    .transform(lambda x: x.rolling(window=5, min_periods=3).std())
)

# --- 2C: Trend Direction (5-year linear slope) ---
# A positive slope means risk is INCREASING. Negative means IMPROVING.
# We use a rolling linear regression coefficient.
from numpy.polynomial.polynomial import polyfit

def rolling_slope(series, window=5, min_periods=3):
    """Compute the slope of a linear fit over a rolling window."""
    result = pd.Series(index=series.index, dtype=float)
    for i in range(len(series)):
        start = max(0, i - window + 1)
        window_data = series.iloc[start:i+1].dropna()
        if len(window_data) >= min_periods:
            x = np.arange(len(window_data))
            # polyfit returns [intercept, slope] for degree 1
            coeffs = polyfit(x, window_data.values, 1)
            result.iloc[i] = coeffs[1]  # slope
    return result

wri['ccrs_trend_5y'] = (
    wri.groupby('iso3')['ccrs']
    .transform(lambda x: rolling_slope(x, window=5, min_periods=3))
)

# --- 2D: Maximum Drawdown (from best-ever score) ---
# In investment analysis, drawdown measures the loss from peak.
# Here it measures: "how much has this country's risk INCREASED
# relative to its best-ever (lowest-risk) year?"
# A country at its all-time worst is in maximum drawdown.
wri['ccrs_running_min'] = (
    wri.groupby('iso3')['ccrs']
    .transform(lambda x: x.expanding().min())
)
wri['ccrs_drawdown'] = wri['ccrs'] - wri['ccrs_running_min']

# --- 2E: Risk Regime Classification ---
# Categorize each country-year into one of three regimes
# based on the 5-year trend and recent volatility.
def classify_regime(row):
    trend = row['ccrs_trend_5y']
    vol = row['ccrs_vol_3y']
    if pd.isna(trend) or pd.isna(vol):
        return 'insufficient_data'
    elif trend < -0.3:
        return 'improving'
    elif trend > 0.3:
        return 'deteriorating'
    else:
        return 'stable'

wri['risk_regime'] = wri.apply(classify_regime, axis=1)

print("=" * 70)
print("TEMPORAL RISK FEATURES COMPUTED")
print("=" * 70)
print(f"New columns: ccrs_yoy, ccrs_yoy_pct, ccrs_vol_3y, ccrs_vol_5y,")
print(f"             ccrs_trend_5y, ccrs_drawdown, risk_regime")

latest = wri[wri['year'] == wri['year'].max()]
print(f"\nRisk regime distribution ({latest['year'].iloc[0]}):")
print(latest['risk_regime'].value_counts().to_string())

print(f"\nMost volatile countries (highest 5-year volatility):")
vol_cols = ['country', 'ccrs', 'ccrs_vol_5y', 'ccrs_trend_5y', 'risk_regime']
print(latest.nlargest(10, 'ccrs_vol_5y')[vol_cols].to_string(index=False))

print(f"\nLargest drawdowns (biggest deterioration from best-ever):")
dd_cols = ['country', 'ccrs', 'ccrs_running_min', 'ccrs_drawdown']
print(latest.nlargest(10, 'ccrs_drawdown')[dd_cols].to_string(index=False))

TEMPORAL RISK FEATURES COMPUTED
New columns: ccrs_yoy, ccrs_yoy_pct, ccrs_vol_3y, ccrs_vol_5y,
             ccrs_trend_5y, ccrs_drawdown, risk_regime

Risk regime distribution (2025):
risk_regime
stable           107
improving         44
deteriorating     42

Most volatile countries (highest 5-year volatility):
                              country  ccrs  ccrs_vol_5y  ccrs_trend_5y   risk_regime
                              Jamaica 32.86         2.26           1.04 deteriorating
                              Hungary 22.63         2.09          -1.04     improving
                               Turkey 47.10         1.95           1.08 deteriorating
                              Lebanon 48.17         1.90           1.12 deteriorating
                              Albania 29.53         1.83          -1.09     improving
                              Armenia 33.16         1.82          -1.11     improving
                              Ecuador 45.08         1.78           1.04 deteriorating

## Compound Risk

Individual dimension scores miss something important: the non-linear danger of multiple weaknesses co-occurring. A country with one elevated risk dimension is manageable; one with four or five elevated simultaneously is qualitatively different — each weakness removes a buffer that would otherwise contain the others. The compound multiplier formalises this with intentionally non-linear amplification: moving from three elevated dimensions to four adds more than moving from two to three, because the failure modes start interacting.

In [21]:
# Count elevated dimensions: how many of the six scores exceed the concern threshold?
# SCCR counts here too — high strategic concentration amplifies fragility risk.
# A spike in even one fragility dimension at a critical node (e.g. Singapore)
# cascades differently than the same spike in a peripheral corridor.
CONCERN_THRESHOLD = 50
risk_cols_for_compound = ['nhes', 'gss_risk', 'irs', 'efs', 'rsbs', 'sccr']
wri['risk_dimensions_elevated'] = (
    wri[risk_cols_for_compound]
    .apply(lambda row: (row > CONCERN_THRESHOLD).sum(), axis=1)
)

# Non-linear amplification: compounding accelerates as more dimensions elevate simultaneously.
# 0–1 elevated: 1.0x  (manageable, buffers still intact)
# 2 elevated:   1.1x
# 3 elevated:   1.25x
# 4 elevated:   1.5x
# 5 elevated:   1.8x
# 6 elevated:   2.0x  (all dimensions elevated — failure modes are interacting)
compound_map = {0: 1.0, 1: 1.0, 2: 1.1, 3: 1.25, 4: 1.5, 5: 1.8, 6: 2.0}
wri['compound_multiplier'] = wri['risk_dimensions_elevated'].map(compound_map)

# Compound-adjusted CCRS
wri['ccrs_compound'] = (wri['ccrs'] * wri['compound_multiplier']).clip(upper=100)

print('=' * 70)
print('COMPOUND RISK SCORE')
print('Tracking 6 dimensions: nhes, gss_risk, irs, efs, rsbs, sccr')
print('=' * 70)
latest = wri[wri['year'] == wri['year'].max()]
print('\nDistribution of elevated risk dimensions:')
print(latest['risk_dimensions_elevated'].value_counts().sort_index().to_string())

print('\nCountries with compound risk (4+ dimensions elevated):')
compound_cols = ['country', 'ccrs', 'ccrs_compound', 'risk_dimensions_elevated',
                 'nhes', 'gss_risk', 'irs', 'efs', 'rsbs', 'sccr']
high_compound = latest[latest['risk_dimensions_elevated'] >= 4]
print(high_compound.nlargest(15, 'ccrs_compound')[compound_cols].to_string(index=False))


COMPOUND RISK SCORE
Tracking 6 dimensions: nhes, gss_risk, irs, efs, rsbs, sccr

Distribution of elevated risk dimensions:
risk_dimensions_elevated
0    63
1    41
2    37
3    37
4    14
5     1

Countries with compound risk (4+ dimensions elevated):
                     country  ccrs  ccrs_compound  risk_dimensions_elevated  nhes  gss_risk   irs   efs  rsbs  sccr
                       Yemen 63.75         100.00                         5 28.63     89.76 57.45 68.56 69.07 72.00
                     Somalia 65.98          98.96                         4 26.32     90.02 67.29 70.90 70.57  5.00
Democratic Republic of Congo 65.02          97.53                         4 27.78     82.41 78.42 70.29 63.17  5.00
                 Afghanistan 62.45          93.67                         4 20.88     84.33 56.09 72.19 75.72  5.00
                    Ethiopia 61.68          92.52                         4 20.27     66.65 68.75 68.02 85.07  5.00
                 South Sudan 60.73          91.09   

In [22]:
# SCCR applied as a post-composite multiplier, not a weighted input.
#
# Direction: higher SCCR = more strategically critical = same direction as CCRS.
# No inversion needed here.
#
# The reason it's a multiplier rather than a feature in CCRS_WEIGHTS:
# SCCR measures disruption consequence; the other five measure disruption probability.
# Averaging them together blurs two distinct analytical questions into one number.
#
# Ceiling is 1.5x (not 2.5x) because the chokepoint multipliers in route aggregation
# already handle geographic amplification at the route level. Stacking both at 2.5x
# would double-count the same signal.

def sccr_multiplier(sccr_score, floor=1.0, ceiling=1.5):
    """
    Scales CCRS upward based on strategic node importance.
    SCCR=0   → 1.0x  (no amplification — country is not a critical node)
    SCCR=50  → 1.25x (moderate amplification)
    SCCR=100 → 1.50x (maximum amplification — irreplaceable chokepoint)
    Linear interpolation between floor and ceiling.
    """
    return floor + (sccr_score / 100) * (ceiling - floor)

wri['sccr_multiplier'] = wri['sccr'].apply(sccr_multiplier)
wri['ccrs_final'] = (wri['ccrs_compound'] * wri['sccr_multiplier']).clip(upper=100)

# ---- Validation ----
print("=" * 70)
print("SCCR MULTIPLIER — IMPACT ADJUSTMENT")
print("Direction: higher SCCR = higher multiplier = higher final risk")
print("=" * 70)

latest = wri[wri['year'] == wri['year'].max()]

print("\nKey validation cases:")
validate = ['Singapore', 'Yemen', 'Somalia', 'Netherlands', 'Chad', 'Japan']
cols = ['country', 'ccrs_compound', 'sccr', 'sccr_multiplier', 'ccrs_final']
mask = latest['country'].isin(validate)
print(latest[mask].sort_values('ccrs_final', ascending=False)[cols].to_string(index=False))

print("\nSingapore directional check:")
sgp = latest[latest['iso3'] == 'SGP']
print(f"  ccrs_compound (probability layer): {sgp['ccrs_compound'].values[0]:.1f}  ← low fragility")
print(f"  sccr (impact layer):               {sgp['sccr'].values[0]:.1f}        ← high consequence")
print(f"  sccr_multiplier:                   {sgp['sccr_multiplier'].values[0]:.2f}x")
print(f"  ccrs_final:                        {sgp['ccrs_final'].values[0]:.1f}        ← correctly elevated")
print(f"  Interpretation: low probability of disruption, but high impact")
print(f"  if disrupted — model now captures both signals independently.")

SCCR MULTIPLIER — IMPACT ADJUSTMENT
Direction: higher SCCR = higher multiplier = higher final risk

Key validation cases:
    country  ccrs_compound  sccr  sccr_multiplier  ccrs_final
    Somalia          98.96  5.00             1.02      100.00
      Yemen         100.00 72.00             1.36      100.00
       Chad          89.04  5.00             1.02       91.26
      Japan          29.01 40.00             1.20       34.81
Netherlands          20.27 58.00             1.29       26.15
  Singapore           9.31 90.00             1.45       13.50

Singapore directional check:
  ccrs_compound (probability layer): 9.3  ← low fragility
  sccr (impact layer):               90.0        ← high consequence
  sccr_multiplier:                   1.45x
  ccrs_final:                        13.5        ← correctly elevated
  Interpretation: low probability of disruption, but high impact
  if disrupted — model now captures both signals independently.


## Route Aggregation

Country-level scores are inputs, not decisions. The actual output of this model is route-level risk — and translating country scores into corridor scores requires a choice about what question you're asking. Four aggregation methods are implemented here, each with a different assumption about how supply chains fail.

The simple weighted average treats all segments proportionally. 

The bottleneck model gives extra weight to the worst single segment, on the reasoning that one failure can block the whole route. 

The chokepoint-adjusted method amplifies scores at strategically irreplaceable nodes. 

The probabilistic approach treats each segment as an independent failure event and estimates the likelihood of at least one disruption occurring across the full route. 

Including all four is deliberate — it shows the sensitivity of the answer to the aggregation assumption, which is a more honest representation than picking one method and presenting it as fact.

In [24]:
# Chokepoint multipliers derived from SCCR rather than hardcoded.
# Previously the two tables were disconnected — a change to SCCR_SCORES
# wouldn't propagate to the route layer automatically. Now they're in sync.
#
# Scale: SCCR=0 → 1.0x, SCCR=100 → 2.5x.
# The 2.5 ceiling matches the previous hardcoded Suez value, keeping
# route score comparisons consistent with earlier analysis.

def sccr_to_chokepoint_multiplier(sccr_score, min_mult=1.0, max_mult=2.5):
    """
    Convert an SCCR score (0–100) to a chokepoint multiplier (1.0–2.5).
    Linear scale: SCCR=0 → 1.0x, SCCR=100 → 2.5x.
    
    Note: sccr_multiplier() in Section 3 uses a 1.0–1.5 ceiling because it
    adjusts the country-level CCRS. This function uses a 1.0–2.5 ceiling
    because it amplifies route-level scores where geographic irreplaceability
    (e.g. the Suez Canal) warrants stronger amplification.
    Different ceilings for different analytical contexts — not an inconsistency.
    """
    return min_mult + (sccr_score / 100) * (max_mult - min_mult)

# Build CHOKEPOINTS dynamically from SCCR_SCORES lookup table
# Only countries that appear in SCCR_SCORES get an amplified multiplier.
# All others default to 1.0 in score_route_chokepoint().
CHOKEPOINTS = {
    iso3: sccr_to_chokepoint_multiplier(score)
    for iso3, score in SCCR_SCORES.items()
}

# Sanity check — compare derived values to previously hardcoded values
print('DERIVED CHOKEPOINT MULTIPLIERS (from SCCR scores)')
print(f'{"ISO3":6s} {"SCCR":>6s} {"Derived":>8s} {"Old hardcoded":>14s}')
print('-' * 40)
old_hardcoded = {'EGY': 2.5, 'SGP': 1.5, 'PAN': 1.8, 'DJI': 2.0,
                 'MYS': 1.5, 'TUR': 1.5, 'OMN': 1.8, 'IRN': 1.8}
for iso3 in ['EGY', 'SGP', 'PAN', 'DJI', 'MYS', 'TUR', 'OMN', 'IRN']:
    sccr_val = SCCR_SCORES.get(iso3, 5)
    derived  = CHOKEPOINTS.get(iso3, 1.0)
    old      = old_hardcoded.get(iso3, '-')
    print(f'{iso3:6s} {sccr_val:6.0f} {derived:8.2f} {str(old):>14s}')
print('\nNote: SGP rises from 1.5 → ~2.35 — a more accurate reflection of')
print('Malacca\'s strategic weight. The old 1.5 was an underestimate.')

# Example routes: Vietnam to Rotterdam
EXAMPLE_ROUTES = {
    'Route A: Suez Corridor': {
        'countries': ['VNM', 'MYS', 'LKA', 'DJI', 'EGY', 'NLD'],
        'distance_shares': [0.15, 0.10, 0.10, 0.15, 0.20, 0.30],},
    'Route B: Pacific-Panama': {
        'countries': ['VNM', 'PHL', 'USA', 'PAN', 'NLD'],
        'distance_shares': [0.15, 0.10, 0.30, 0.15, 0.30],},
    'Route C: Cape of Good Hope': {
        'countries': ['VNM', 'MYS', 'LKA', 'MOZ', 'ZAF', 'NLD'],
        'distance_shares': [0.10, 0.08, 0.08, 0.15, 0.24, 0.35],
    },
}


DERIVED CHOKEPOINT MULTIPLIERS (from SCCR scores)
ISO3     SCCR  Derived  Old hardcoded
----------------------------------------
EGY        92     2.38            2.5
SGP        90     2.35            1.5
PAN        85     2.27            1.8
DJI        78     2.17            2.0
MYS        68     2.02            1.5
TUR        65     1.98            1.5
OMN        60     1.90            1.8
IRN        60     1.90            1.8

Note: SGP rises from 1.5 → ~2.35 — a more accurate reflection of
Malacca's strategic weight. The old 1.5 was an underestimate.


In [25]:
# Four aggregation approaches — each answers a different version of "how risky is this route?"
# All use ccrs_final (fragility × SCCR impact) rather than ccrs_compound,
# so strategic node importance is reflected in the route scores.

def score_route_simple_avg(route_countries, distance_shares, risk_df, year, score_col='ccrs_final'):
    """Approach 1: Distance-weighted average of country risk scores."""
    scores = []
    for iso3, share in zip(route_countries, distance_shares):
        row = risk_df[(risk_df['iso3'] == iso3) & (risk_df['year'] == year)]
        if len(row):
            scores.append(row[score_col].values[0] * share)
    return sum(scores) / sum(distance_shares) if scores else None


def score_route_bottleneck(route_countries, distance_shares, risk_df, year,
                           score_col='ccrs_final', lam=0.4):
    """Approach 2: Blend of weighted average and maximum single-country risk.
    lambda=0 → pure average; lambda=1 → pure worst-case."""
    country_scores = []
    for iso3 in route_countries:
        row = risk_df[(risk_df['iso3'] == iso3) & (risk_df['year'] == year)]
        if len(row):
            country_scores.append(row[score_col].values[0])
    if not country_scores:
        return None
    avg = score_route_simple_avg(route_countries, distance_shares, risk_df, year, score_col)
    max_score = max(country_scores)
    return lam * max_score + (1 - lam) * avg


def score_route_chokepoint(route_countries, distance_shares, risk_df, year,
                           chokepoints=CHOKEPOINTS, score_col='ccrs_final', lam=0.4):
    """Approach 3: Chokepoint-weighted with bottleneck correction."""
    weighted_scores = []
    total_weight = 0
    max_score = 0
    for iso3, share in zip(route_countries, distance_shares):
        row = risk_df[(risk_df['iso3'] == iso3) & (risk_df['year'] == year)]
        if len(row):
            score = row[score_col].values[0]
            multiplier = chokepoints.get(iso3, 1.0)
            effective_weight = share * multiplier
            weighted_scores.append(score * effective_weight)
            total_weight += effective_weight
            max_score = max(max_score, score * multiplier)
    if not weighted_scores:
        return None
    choke_avg = sum(weighted_scores) / total_weight
    return lam * max_score + (1 - lam) * choke_avg


def score_route_probabilistic(route_countries, risk_df, year,
                              score_col='ccrs_final'):
    """Approach 4: Probabilistic cascade — probability of at least one disruption.
    Interprets risk score / 100 as disruption probability."""
    prob_all_fine = 1.0
    for iso3 in route_countries:
        row = risk_df[(risk_df['iso3'] == iso3) & (risk_df['year'] == year)]
        if len(row):
            p_disruption = np.clip(row[score_col].values[0] / 100, 0.01, 0.99)
            prob_all_fine *= (1 - p_disruption)
    return (1 - prob_all_fine) * 100


In [26]:
# --- 4C: Score the example routes across all four approaches ---

print('=' * 70)
print('ROUTE RISK COMPARISON — Vietnam to Rotterdam')
print('Using ccrs_final (fragility probability × SCCR impact multiplier)')
print('=' * 70)

latest_year = wri['year'].max()

for route_name, route_info in EXAMPLE_ROUTES.items():
    countries = route_info['countries']
    shares = route_info['distance_shares']

    print(f'\n{route_name}')
    print(f'  Path: {" → ".join(countries)}')

    # Show individual country scores using ccrs_final
    for iso3 in countries:
        row = wri[(wri['iso3'] == iso3) & (wri['year'] == latest_year)]
        if len(row):
            choke = CHOKEPOINTS.get(iso3, 1.0)
            choke_flag = f' ⚠ CHOKEPOINT ×{choke:.2f}' if choke > 1.0 else ''
            print(f'    {row["country"].values[0]:20s} ccrs_final={row["ccrs_final"].values[0]:5.1f}{choke_flag}')

    # All four aggregation approaches
    s1 = score_route_simple_avg(countries, shares, wri, latest_year)
    s2 = score_route_bottleneck(countries, shares, wri, latest_year, lam=0.4)
    s3 = score_route_chokepoint(countries, shares, wri, latest_year, lam=0.4)
    s4 = score_route_probabilistic(countries, wri, latest_year)

    print(f'  Scores:')
    print(f'    Simple Weighted Avg:  {s1:.1f}' if s1 else '    Simple Avg: N/A')
    print(f'    Bottleneck (λ=0.4):  {s2:.1f}' if s2 else '    Bottleneck: N/A')
    print(f'    Chokepoint+Bottle:   {s3:.1f}' if s3 else '    Chokepoint: N/A')
    print(f'    Probabilistic:       {s4:.1f}' if s4 else '    Probabilistic: N/A')


ROUTE RISK COMPARISON — Vietnam to Rotterdam
Using ccrs_final (fragility probability × SCCR impact multiplier)

Route A: Suez Corridor
  Path: VNM → MYS → LKA → DJI → EGY → NLD
    Viet Nam             ccrs_final= 41.8
    Malaysia             ccrs_final= 43.9 ⚠ CHOKEPOINT ×2.02
    Sri Lanka            ccrs_final= 40.1
    Djibouti             ccrs_final= 73.4 ⚠ CHOKEPOINT ×2.17
    Egypt                ccrs_final= 85.7 ⚠ CHOKEPOINT ×2.38
    Netherlands          ccrs_final= 26.2 ⚠ CHOKEPOINT ×1.87
  Scores:
    Simple Weighted Avg:  50.7
    Bottleneck (λ=0.4):  64.7
    Chokepoint+Bottle:   114.2
    Probabilistic:       99.5

Route B: Pacific-Panama
  Path: VNM → PHL → USA → PAN → NLD
    Viet Nam             ccrs_final= 41.8
    Philippines          ccrs_final= 71.2
    United States of America ccrs_final= 33.6 ⚠ CHOKEPOINT ×1.68
    Panama               ccrs_final= 53.1 ⚠ CHOKEPOINT ×2.27
    Netherlands          ccrs_final= 26.2 ⚠ CHOKEPOINT ×1.87
  Scores:
    Simple Weighted A

## Machine Learning — A Second Opinion on the Weights

The five engineered features and their weights are explicit analytical choices. They're transparent, but potentially miscalibrated in ways that are hard to see from inside the model. The ML layer acts as a validation step: if Random Forest feature importances roughly agree with the CCRS weights, that corroborates the design. If they diverge significantly, that's worth investigating.

Both models are trained on a temporal split rather than a random split. Training on 2020 data and testing on 2015 data is data leakage — the model would be evaluated on a period it has already seen in reverse. TimeSeriesSplit enforces the correct direction.

In [28]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import sklearn.metrics as metrics

In [29]:
# ML target: risk tiers derived from ccrs_final (the complete score including SCCR).
# Using ccrs_compound instead would train the model without any awareness of
# strategic node importance — precisely the gap the SCCR feature was built to fill.

def assign_risk_tier(score):
    if score < 20:
        return 'Low'
    elif score < 40:
        return 'Medium-Low'
    elif score < 60:
        return 'Medium-High'
    else:
        return 'High'

wri['risk_tier'] = wri['ccrs_final'].apply(assign_risk_tier)

print('=' * 70)
print('ML TARGET: Risk Tier Distribution (from ccrs_final)')
print('=' * 70)
print(wri['risk_tier'].value_counts().sort_index().to_string())

# Spot-check Singapore: with SCCR applied, does its tier shift?
sgp = wri[(wri['iso3'] == 'SGP') & (wri['year'] == wri['year'].max())]
if len(sgp):
    print(f'\nSingapore tier check:')
    print(f'  ccrs_compound: {sgp["ccrs_compound"].values[0]:.1f} → tier without SCCR: '
          f'{assign_risk_tier(sgp["ccrs_compound"].values[0])}')
    print(f'  ccrs_final:    {sgp["ccrs_final"].values[0]:.1f} → tier with SCCR:    '
          f'{sgp["risk_tier"].values[0]}')


ML TARGET: Risk Tier Distribution (from ccrs_final)
risk_tier
High            931
Low             571
Medium-High    1523
Medium-Low     1993

Singapore tier check:
  ccrs_compound: 9.3 → tier without SCCR: Low
  ccrs_final:    13.5 → tier with SCCR:    Low


In [30]:
# Feature matrix includes both the fragility layer (5 dimensions) and the impact layer (SCCR).
# SCCR is static across years — it reflects structural geography, not annual measurement.
# The model correctly learns that certain nodes are systematically more consequential.

ML_FEATURES = [
    # Five core fragility dimensions (probability layer)
    'nhes', 'gss_risk', 'irs', 'efs', 'rsbs',
    # Strategic impact dimension
    'sccr', 'sccr_multiplier',
    # Temporal context
    'ccrs_yoy', 'ccrs_vol_3y', 'ccrs_trend_5y', 'ccrs_drawdown',
    # Compound risk indicator (now counts 6 dimensions including sccr)
    'risk_dimensions_elevated',
]

# Drop rows with NaN in features (early years lack rolling calculations)
ml_data = wri.dropna(subset=ML_FEATURES).copy()

X = ml_data[ML_FEATURES]
y = LabelEncoder().fit_transform(ml_data['risk_tier'])

le = LabelEncoder().fit(ml_data['risk_tier'])
print(f'\nML dataset: {X.shape[0]} observations × {X.shape[1]} features')
print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'Year range after dropping NaN: {ml_data["year"].min()}-{ml_data["year"].max()}')
print(f'\nFeature list ({len(ML_FEATURES)} total):')
for f in ML_FEATURES:
    print(f'  {f}')



ML dataset: 4632 observations × 12 features
Label mapping: {'High': 0, 'Low': 1, 'Medium-High': 2, 'Medium-Low': 3}
Year range after dropping NaN: 2002-2025

Feature list (12 total):
  nhes
  gss_risk
  irs
  efs
  rsbs
  sccr
  sccr_multiplier
  ccrs_yoy
  ccrs_vol_3y
  ccrs_trend_5y
  ccrs_drawdown
  risk_dimensions_elevated


In [31]:
# Temporal split — not random.
# A random split would let the model train on future observations relative to
# its test period, which is data leakage. Pre-2022 trains, 2022+ tests.

# Simple temporal split: train on years before 2022, test on 2022+
train_mask = ml_data['year'] < 2022
test_mask = ml_data['year'] >= 2022

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"\nTemporal split:")
print(f"  Train: {X_train.shape[0]} observations (years < 2022)")
print(f"  Test:  {X_test.shape[0]} observations (years >= 2022)")


Temporal split:
  Train: 3860 observations (years < 2022)
  Test:  772 observations (years >= 2022)


In [32]:
# Random Forest — interpretable baseline model
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=20,    # keeps the model from overfitting to small-country outliers
    class_weight='balanced', # low-risk countries are overrepresented; this corrects for it
    random_state=42
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("=" * 70)
print("RANDOM FOREST — Risk Tier Classification")
print("=" * 70)
print(classification_report(y_test, rf_pred, target_names=le.classes_))

# Feature importances serve as a cross-check on the hand-crafted CCRS weights
rf_importance = pd.DataFrame({
    'feature': ML_FEATURES,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance (Random Forest):")
for _, row in rf_importance.iterrows():
    bar = '█' * int(row['importance'] * 80)
    print(f"  {row['feature']:28s} {row['importance']:.3f} {bar}")

RANDOM FOREST — Risk Tier Classification
              precision    recall  f1-score   support

        High       0.99      0.97      0.98       134
         Low       0.88      0.96      0.92        81
 Medium-High       0.90      0.96      0.93       220
  Medium-Low       0.97      0.91      0.94       337

    accuracy                           0.94       772
   macro avg       0.93      0.95      0.94       772
weighted avg       0.94      0.94      0.94       772


Feature Importance (Random Forest):
  gss_risk                     0.228 ██████████████████
  risk_dimensions_elevated     0.216 █████████████████
  rsbs                         0.192 ███████████████
  irs                          0.131 ██████████
  nhes                         0.090 ███████
  efs                          0.079 ██████
  ccrs_drawdown                0.018 █
  sccr_multiplier              0.017 █
  sccr                         0.017 █
  ccrs_vol_3y                  0.005 
  ccrs_trend_5y                

In [33]:
# Gradient Boosting — stronger model, less interpretable
gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    min_samples_leaf=20,
    random_state=42
)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

print("\n" + "=" * 70)
print("GRADIENT BOOSTING — Risk Tier Classification")
print("=" * 70)
print(classification_report(y_test, gb_pred, target_names=le.classes_))

# High agreement between RF and GB suggests the classification is stable
# across different learning algorithms — a useful signal of robustness.
agreement = (rf_pred == gb_pred).mean()
print(f"\nModel agreement (RF vs GB): {agreement:.1%}")
print("If agreement is high, our classification is robust across model types.")


GRADIENT BOOSTING — Risk Tier Classification
              precision    recall  f1-score   support

        High       0.99      0.97      0.98       134
         Low       0.99      0.93      0.96        81
 Medium-High       0.94      0.97      0.95       220
  Medium-Low       0.96      0.97      0.97       337

    accuracy                           0.96       772
   macro avg       0.97      0.96      0.96       772
weighted avg       0.96      0.96      0.96       772


Model agreement (RF vs GB): 94.3%
If agreement is high, our classification is robust across model types.


## LSTM — Sequence-Aware Risk Forecasting

The Random Forest treats each country-year observation independently. The LSTM treats each as a step in a sequence, which is more structurally honest: risk trajectories have momentum, and a country deteriorating for three consecutive years is more likely to continue than one that peaked and is recovering.

One constraint worth being upfront about: with ~193 countries and roughly 20 usable years each, the training set is around 3,800 sequences — small for deep learning. The value of the LSTM here is architectural rather than purely predictive: it demonstrates sequence modelling and temporal data structure, and it sets up a clear path to improvement (adding ACLED conflict event data and FRED macro indicators would substantially improve sequence quality and input diversity).

In [35]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler

print("TensorFlow version:", tf.__version__)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow version: 2.13.0


In [36]:
# Convert the panel data into LSTM-ready sequences.
# For each country: 5 consecutive years of features → next year's CCRS.

LOOKBACK = 5  # 5-year window: long enough to capture trend direction, short enough to maximise sequences
FORECAST_FEATURES = ['nhes', 'gss_risk', 'irs', 'efs', 'rsbs',
                     'ccrs_vol_3y', 'risk_dimensions_elevated']

# Neural networks are sensitive to input scale; normalise everything to 0-1.
scaler = MinMaxScaler()

def create_sequences(df, lookback=LOOKBACK, features=FORECAST_FEATURES, target='ccrs'):
    """Convert panel data into LSTM-ready sequences.
    Returns X with shape (samples, lookback, n_features) and y with shape (samples,).
    """
    X_seq, y_seq, meta = [], [], []

    for iso3, group in df.groupby('iso3'):
        group = group.sort_values('year').dropna(subset=features + [target])
        if len(group) < lookback + 1:
            continue  # Need at least lookback+1 years

        values = group[features].values
        targets = group[target].values
        years = group['year'].values
        country = group['country'].values[0]

        for i in range(lookback, len(group)):
            X_seq.append(values[i-lookback:i])  # Previous N years
            y_seq.append(targets[i])             # Next year's score
            meta.append({'iso3': iso3, 'country': country, 'year': years[i]})

    return np.array(X_seq), np.array(y_seq), pd.DataFrame(meta)

# Scale features before creating sequences
wri_scaled = wri.copy()
wri_scaled[FORECAST_FEATURES] = scaler.fit_transform(
    wri[FORECAST_FEATURES].fillna(0)
)

X_seq, y_seq, seq_meta = create_sequences(wri_scaled)

print(f"LSTM sequences created:")
print(f"  X shape: {X_seq.shape} (samples, lookback={LOOKBACK}, features={len(FORECAST_FEATURES)})")
print(f"  y shape: {y_seq.shape}")
print(f"  Countries with sequences: {seq_meta['iso3'].nunique()}")

LSTM sequences created:
  X shape: (4053, 5, 7) (samples, lookback=5, features=7)
  y shape: (4053,)
  Countries with sequences: 193


In [37]:
# --- 6B: Temporal train/test split for LSTM ---
# Same principle: train on earlier years, test on recent years.

train_idx = seq_meta['year'] < 2022
test_idx = seq_meta['year'] >= 2022

X_train_lstm = X_seq[train_idx]
y_train_lstm = y_seq[train_idx]
X_test_lstm = X_seq[test_idx]
y_test_lstm = y_seq[test_idx]

# Scale the target variable too (CCRS ranges 0-100)
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train_lstm.reshape(-1, 1)).flatten()
y_test_scaled = y_scaler.transform(y_test_lstm.reshape(-1, 1)).flatten()

print(f"LSTM train: {X_train_lstm.shape[0]} sequences")
print(f"LSTM test:  {X_test_lstm.shape[0]} sequences")

LSTM train: 3281 sequences
LSTM test:  772 sequences


In [38]:
# LSTM architecture — two stacked layers with dropout regularisation
model = Sequential([
    # First LSTM layer processes the 5-year input sequence
    LSTM(64, input_shape=(LOOKBACK, len(FORECAST_FEATURES)),
         return_sequences=True),
    Dropout(0.2),  # dropout regularisation — the dataset is small, overfitting is a real risk

    # Second LSTM layer picks up higher-order patterns across the sequence
    LSTM(32, return_sequences=False),
    Dropout(0.2),

    # Dense layers compress LSTM output to a single risk score
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')  # output 0-1; rescaled to CCRS range (0-100) after prediction-100
])

model.compile(
    optimizer='adam',
    loss='mse',             # Mean squared error for regression
    metrics=['mae']         # Mean absolute error for interpretability
)

print(model.summary())

# Train with early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_lstm, y_train_scaled,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0  # Set to 1 to see training progress
)

print(f"\nTraining complete. Best epoch: {early_stop.stopped_epoch - 10 + 1}")
print(f"Final val_loss: {min(history.history['val_loss']):.4f}")

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 5, 64)             18432     
                                                                 
 dropout (Dropout)           (None, 5, 64)             0         
                                                                 
 lstm_1 (LSTM)               (None, 32)                12416     
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense (Dense)               (None, 16)                528       
                                                                 
 dense_1 (Dense)             (None, 1)                 17        
                                                                 
Total params: 31393 (122.63 KB)
Trainable params: 31393 

In [39]:
# --- 6D: Evaluate LSTM forecasting accuracy ---
y_pred_scaled = model.predict(X_test_lstm, verbose=0).flatten()
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

# Metrics
mae = np.mean(np.abs(y_pred - y_test_lstm))
rmse = np.sqrt(np.mean((y_pred - y_test_lstm) ** 2))
# R-squared: how much variance does the model explain?
ss_res = np.sum((y_test_lstm - y_pred) ** 2)
ss_tot = np.sum((y_test_lstm - np.mean(y_test_lstm)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print("=" * 70)
print("LSTM FORECASTING RESULTS")
print("=" * 70)
print(f"Mean Absolute Error:  {mae:.2f} points (on 0-100 scale)")
print(f"Root Mean Sq Error:   {rmse:.2f} points")
print(f"R-squared:            {r2:.3f}")
print(f"\nInterpretation: The model predicts next-year CCRS within ±{mae:.1f}")
print(f"points on average. For a 0-100 scale, this is {'good' if mae < 5 else 'moderate' if mae < 10 else 'needs improvement'}.")

# Show predictions vs actuals for specific countries
test_with_pred = seq_meta[test_idx].copy()
test_with_pred['actual'] = y_test_lstm
test_with_pred['predicted'] = y_pred
test_with_pred['error'] = y_pred - y_test_lstm

print(f"\nSample predictions (2022+):")
sample_countries = ['SGP', 'YEM', 'JPN', 'NGA', 'BRA', 'DEU']
for iso3 in sample_countries:
    rows = test_with_pred[test_with_pred['iso3'] == iso3].tail(2)
    if len(rows):
        for _, r in rows.iterrows():
            print(f"  {r['country']:15s} {int(r['year'])}: "
                  f"actual={r['actual']:.1f}, predicted={r['predicted']:.1f}, "
                  f"error={r['error']:+.1f}")

LSTM FORECASTING RESULTS
Mean Absolute Error:  0.95 points (on 0-100 scale)
Root Mean Sq Error:   1.26 points
R-squared:            0.990

Interpretation: The model predicts next-year CCRS within ±0.9
points on average. For a 0-100 scale, this is good.

Sample predictions (2022+):
  Singapore       2024: actual=9.3, predicted=9.2, error=-0.1
  Singapore       2025: actual=9.3, predicted=9.3, error=-0.0
  Yemen           2024: actual=63.8, predicted=61.8, error=-1.9
  Yemen           2025: actual=63.8, predicted=62.2, error=-1.5
  Japan           2024: actual=29.0, predicted=29.2, error=+0.2
  Japan           2025: actual=29.0, predicted=28.9, error=-0.1
  Nigeria         2024: actual=55.1, predicted=54.0, error=-1.1
  Nigeria         2025: actual=55.1, predicted=54.3, error=-0.7
  Brazil          2024: actual=39.0, predicted=38.2, error=-0.8
  Brazil          2025: actual=39.0, predicted=38.7, error=-0.3
  Germany         2024: actual=19.7, predicted=20.5, error=+0.9
  Germany         

In [40]:
# Export

export_cols = [
    # Row identity
    'country', 'iso3', 'year',
    # Five fragility features (probability layer)
    'nhes', 'gss', 'gss_risk', 'irs', 'efs', 'rsbs',
    # Strategic impact feature
    'sccr', 'sccr_multiplier',
    # Three-stage composite: each step adds one layer of adjustment
    'ccrs',           # base: 5-feature weighted composite
    'ccrs_compound',  # + compounding amplification
    'ccrs_final',     # + SCCR impact multiplier (definitive score)efinitive score)
    'compound_multiplier', 'risk_dimensions_elevated',
    # Temporal features
    'ccrs_yoy', 'ccrs_yoy_pct', 'ccrs_vol_3y', 'ccrs_vol_5y',
    'ccrs_trend_5y', 'ccrs_drawdown', 'risk_regime',
    # ML classification (based on ccrs_final)
    'risk_tier',
]

export_cols = [c for c in export_cols if c in wri.columns]
wri_features = wri[export_cols].copy()

OUTPUT_PATH = './'
wri_features.to_csv(f'{OUTPUT_PATH}country_risk_features.csv', index=False)

route_results = []
for year in range(2015, wri['year'].max() + 1):
    for route_name, route_info in EXAMPLE_ROUTES.items():
        countries = route_info['countries']
        shares = route_info['distance_shares']
        route_results.append({
            'year': year,
            'route': route_name,
            'simple_avg':   score_route_simple_avg(countries, shares, wri, year),
            'bottleneck':   score_route_bottleneck(countries, shares, wri, year),
            'chokepoint':   score_route_chokepoint(countries, shares, wri, year),
            'probabilistic': score_route_probabilistic(countries, wri, year),
        })

route_df = pd.DataFrame(route_results)
route_df.to_csv(f'{OUTPUT_PATH}route_risk_scores.csv', index=False)

print('=' * 70)
print('PHASE 2 EXPORTS')
print('=' * 70)
print(f'1. country_risk_features.csv — {wri_features.shape[0]} rows × {wri_features.shape[1]} cols')
print(f'   Scores exported: ccrs (probability) → ccrs_compound → ccrs_final (with SCCR impact)')
print(f'2. route_risk_scores.csv     — {route_df.shape[0]} rows × {route_df.shape[1]} cols')
print(f'   Routes scored using ccrs_final — both fragility and strategic impact embedded')
print(f'\nPhase 2 complete.')


PHASE 2 EXPORTS
1. country_risk_features.csv — 5018 rows × 24 cols
   Scores exported: ccrs (probability) → ccrs_compound → ccrs_final (with SCCR impact)
2. route_risk_scores.csv     — 33 rows × 6 cols
   Routes scored using ccrs_final — both fragility and strategic impact embedded

Phase 2 complete.
